In [1]:
from pathlib import Path
import numpy as np
import os
import pandas as pd
import re
import time

In [2]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../')
print(sys.path)

['/home/thule/miniconda3/envs/open-clip-training/lib/python311.zip', '/home/thule/miniconda3/envs/open-clip-training/lib/python3.11', '/home/thule/miniconda3/envs/open-clip-training/lib/python3.11/lib-dynload', '', '/home/thule/miniconda3/envs/open-clip-training/lib/python3.11/site-packages', '../']


In [3]:
# check schema of a .csv file
df = pd.read_csv(f"../data/can-ml/can-ml/2017-subaru-forester/post-attack-labeled/combined-attacks/double-1.csv")
print(df.head())

      timestamp arbitration_id        data_field  attack
0  1.672531e+09            140  0001240828282720       0
1  1.672531e+09            141  E32A3F2B24880002       0
2  1.672531e+09            144  0003773830A80340       0
3  1.672531e+09            148  0B10218841006605       0
4  1.672531e+09            149  160C3F29AA1E0000       0


In [ ]:
import os
import pandas as pd

# Define the directory path for the attack folders
input_dir = '../data/can-ml/can-ml/2017-subaru-forester/post-attack-labeled'  # Replace with your directory path
output_dir = '../data/can-ml/can-ml/2017-subaru-forester/090726/merged'  # Replace with the output directory path for merged files

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# List of attack types (folder names)
attack_types = [
                'combined-attacks', 
                'DoS-attacks', 
                'fuzzing-attacks', 'gear-attacks',
                'interval-attacks', 'rpm-attacks', 'speed-attacks', 'standstill-attacks', 'systematic-attacks'
                ]

# Loop through each attack type
for attack_type in attack_types:
    attack_folder = os.path.join(input_dir, attack_type)
    
    # Check if the folder exists
    if not os.path.isdir(attack_folder):
        print(f"Folder {attack_folder} does not exist, skipping.")
        continue
    
    # Initialize a list to store DataFrames
    df_list = []
    
    # Loop through each CSV file in the folder
    for file_name in os.listdir(attack_folder):
        if file_name.endswith('.csv'):
            file_path = os.path.join(attack_folder, file_name)
            
            # Read the CSV file
            df = pd.read_csv(file_path)
            
            # Data preprocessing
            # Convert arbitration_id to integer (decimal) if it's a valid value
            df['arbitration_id'] = df['arbitration_id'].apply(lambda x: int(x, 16) if isinstance(x, str) else x)
            
            # Convert data_field to binary format with fixed length of 64 bits if it's a valid string
            df['data_field_binary'] = df['data_field'].apply(lambda x: bin(int(x, 16))[2:].zfill(64) if isinstance(x, str) else x)

            df['data_field_gray'] = df['data_field'].apply(
                lambda x: ''.join(format((b := int(x.ljust(16, '0')[i:i+2], 16)) ^ (b >> 1), '08b') for i in range(0, 16, 2))
                if isinstance(x, str) else x
            )
            
            # Remove rows with any invalid values
            df = df[['timestamp', 'arbitration_id', 'data_field', 'data_field_binary', 'data_field_gray', 'attack']].dropna()

            # Add the DataFrame to the list
            df_list.append(df)
    
    # Merge all DataFrames in the list
    merged_df = pd.concat(df_list, ignore_index=True)
    
    # Save the merged DataFrame to a new CSV file
    attack_type_cleaned = attack_type.replace("-attacks", "")
    output_file = os.path.join(output_dir, f'{attack_type_cleaned}.csv')
    merged_df.to_csv(output_file, index=False)
    
    print(f"Merged {attack_type_cleaned} saved to {output_file}")


Merged DoS saved to ../data/can-ml/can-ml/2017-subaru-forester/070726-dos/merged/DoS.csv


In [12]:
import os
import pandas as pd

pd.set_option('display.max_rows', None)      # Show all rows
pd.set_option('display.max_colwidth', None)  # Show full text width

# Directory path containing CSV files
# /home/hieutt/UniCon/data/can-ml/2017-subaru-forester/merged
input_dir = '../data/can-ml/can-ml/2017-subaru-forester/070726-dos/merged'  # Change the path if needed

# List of CSV files in the directory
files = [f for f in os.listdir(input_dir) if f.endswith('.csv')]

# Function to check each file
def check_data(file_path):
    print(f"\nChecking file: {file_path}")
    
    # Read the file
    df = pd.read_csv(file_path)
    
    # Display the shape of the DataFrame
    print("Shape:", df.shape)
    
    # Display the first 5 rows
    print("Head:")
    display(df.head())
    
    # Check for NaN values
    nan_count = df.isna().sum().sum()
    if nan_count > 0:
        print("Warning: There are NaN values in the data.")
        print(df.isna().sum())
    else:
        print("No NaN values found.")
    
    # Check data types
    print("\nData types:")
    print(df.dtypes)
    
    # Check label distribution if 'label' column exists
    if 'label' in df.columns:
        print("\nLabel distribution:")
        print(df['label'].value_counts())
    
    # Descriptive statistics for numeric columns
    print("\nDescriptive statistics for numeric columns:")
    print(df.describe())
    
# Iterate over each file and check
for file in files:
    check_data(os.path.join(input_dir, file))



Checking file: ../data/can-ml/can-ml/2017-subaru-forester/070726-dos/merged/DoS.csv
Shape: (2297897, 6)
Head:


,timestamp,arbitration_id,data_field,data_field_binary,data_field_gray,attack
0,1.672531e+09,342,0180000000000020,110000000000000000000000000000000000000000000000000100000,111000000000000000000000000000000000000000000000000110000,0
1,1.672531e+09,320,0004A809D2D2FF20,100101010000000100111010010110100101111111100100000,110111111000000110110111011101110111000000000110000,0
2,1.672531e+09,321,3A30872FA8C90002,11101000110000100001110010111110101000110010010000000000000010,10011100101000110001000011100011111100101011010000000000000011,0
3,1.672531e+09,864,401489833170004D,100000000010100100010011000001100110001011100000000000001001101,110000000011110110011011100001000101001010010000000000001101011,0
4,1.672531e+09,865,003A00D408000000,111010000000001101010000001000000000000000000000000000,100111000000001011111000001100000000000000000000000000,0


No NaN values found.

Data types:
timestamp            float64
arbitration_id         int64
data_field               str
data_field_binary     object
data_field_gray       object
attack                 int64
dtype: object

Descriptive statistics for numeric columns:
          timestamp  arbitration_id        attack
count  2.297897e+06    2.297897e+06  2.297897e+06
mean   1.672532e+09    4.054529e+02  2.254148e-02
std    2.930624e+02    3.377873e+02  1.484364e-01
min    1.672531e+09    0.000000e+00  0.000000e+00
25%    1.672531e+09    2.100000e+02  0.000000e+00
50%    1.672531e+09    3.280000e+02  0.000000e+00
75%    1.672532e+09    3.560000e+02  0.000000e+00
max    1.672532e+09    1.994000e+03  1.000000e+00


In [13]:
import os
import pandas as pd

# Define the directory path for the attack folders
input_dir = '../data/can-ml/can-ml/2017-subaru-forester/post-attack-labeled'  # Replace with your directory path
output_dir = '../data/can-ml/can-ml/2017-subaru-forester/220626/merged'  # Replace with the output directory path for merged files

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# List of attack types (folder names)
attack_types = ['combined-attacks', 
                'DoS-attacks', 'fuzzing-attacks', 'gear-attacks',
                'interval-attacks', 'rpm-attacks', 'speed-attacks', 'standstill-attacks', 'systematic-attacks'
                ]

# Loop through each attack type
for attack_type in attack_types:
    attack_folder = os.path.join(input_dir, attack_type)
    
    # Check if the folder exists
    if not os.path.isdir(attack_folder):
        print(f"Folder {attack_folder} does not exist, skipping.")
        continue
    
    # Initialize a list to store DataFrames
    df_list = []
    
    # Loop through each CSV file in the folder
    for file_name in os.listdir(attack_folder):
        if file_name.endswith('.csv'):
            file_path = os.path.join(attack_folder, file_name)
            
            # Read the CSV file
            df = pd.read_csv(file_path)
            # preprocessing 220626
            data_hex = (
                df["data_field"]
                .fillna("")
                .astype(str)
                .str.strip()
                .str.upper()
                .str.replace(r"\s+", "", regex=True)
                .str[:16]
                .str.pad(width=16, side="right", fillchar="0")
            )

            for i in range(8):
                byte_hex = data_hex.str.slice(i * 2, i * 2 + 2)
                df[f"data_{i}"] = byte_hex.apply(lambda x: int(x, 16) / 255.0)
            
            # print(df.head())

            # end preprocessing 220626



            # print(df.head())
            # short_rows = df[df["data_field"].astype(str).str.len() < 8]

            # print(short_rows.head(20))
            
            # Data preprocessing
            # Convert arbitration_id to integer (decimal) if it's a valid value
            # df['arbitration_id'] = df['arbitration_id'].apply(lambda x: int(x, 16) if isinstance(x, str) else x)
            
            # # Convert data_field to binary format with fixed length of 64 bits if it's a valid string
            # df['data_field'] = df['data_field'].apply(lambda x: bin(int(x, 16))[2:].zfill(64) if isinstance(x, str) else x)
            
            # Remove rows with any invalid values
            df = df[['timestamp', 'arbitration_id', 'attack', 'data_0', 'data_1', 
                     'data_2', 'data_3', 'data_4', 'data_5', 'data_6', 'data_7']].dropna()

            # Add the DataFrame to the list
            df_list.append(df)
    
    # Merge all DataFrames in the list
    merged_df = pd.concat(df_list, ignore_index=True)
    
    # Save the merged DataFrame to a new CSV file
    attack_type_cleaned = attack_type.replace("-attacks", "")
    output_file = os.path.join(output_dir, f'{attack_type_cleaned}.csv')
    merged_df.to_csv(output_file, index=False)
    
    print(f"Merged {attack_type_cleaned} saved to {output_file}")


Merged combined saved to ../data/can-ml/can-ml/2017-subaru-forester/220626/merged/combined.csv
Merged DoS saved to ../data/can-ml/can-ml/2017-subaru-forester/220626/merged/DoS.csv
Merged fuzzing saved to ../data/can-ml/can-ml/2017-subaru-forester/220626/merged/fuzzing.csv
Merged gear saved to ../data/can-ml/can-ml/2017-subaru-forester/220626/merged/gear.csv
Merged interval saved to ../data/can-ml/can-ml/2017-subaru-forester/220626/merged/interval.csv
Merged rpm saved to ../data/can-ml/can-ml/2017-subaru-forester/220626/merged/rpm.csv
Merged speed saved to ../data/can-ml/can-ml/2017-subaru-forester/220626/merged/speed.csv
Merged standstill saved to ../data/can-ml/can-ml/2017-subaru-forester/220626/merged/standstill.csv
Merged systematic saved to ../data/can-ml/can-ml/2017-subaru-forester/220626/merged/systematic.csv


In [5]:
# Define the directory path
directory_path = '../data/can-ml/can-ml/2017-subaru-forester/220626/merged/interval.csv'

df = pd.read_csv(directory_path)
# Display the first few rows of the combined DataFrame
filtered_df = df[df['attack'] != 0]
filtered_df

,timestamp,arbitration_id,attack,data_0,data_1,data_2,data_3,data_4,data_5,data_6,data_7
12990,1.672531e+09,0D4,1,0.207843,0.003922,0.203922,0.003922,0.203922,0.003922,0.207843,0.003922
13024,1.672531e+09,0D4,1,0.207843,0.003922,0.203922,0.003922,0.203922,0.003922,0.207843,0.003922
13055,1.672531e+09,0D4,1,0.207843,0.003922,0.203922,0.003922,0.203922,0.003922,0.207843,0.003922
13086,1.672531e+09,0D4,1,0.207843,0.003922,0.203922,0.003922,0.203922,0.003922,0.207843,0.003922
13118,1.672531e+09,0D4,1,0.207843,0.003922,0.203922,0.003922,0.203922,0.003922,0.207843,0.003922
...,...,...,...,...,...,...,...,...,...,...,...
2623397,1.672532e+09,0D4,1,0.207843,0.003922,0.203922,0.003922,0.203922,0.003922,0.207843,0.003922
2623425,1.672532e+09,0D4,1,0.207843,0.003922,0.203922,0.003922,0.203922,0.003922,0.207843,0.003922
2623426,1.672532e+09,0D4,1,0.207843,0.003922,0.203922,0.003922,0.203922,0.003922,0.207843,0.003922
2623427,1.672532e+09,0D4,1,0.207843,0.003922,0.203922,0.003922,0.203922,0.003922,0.207843,0.003922


In [6]:
df['arbitration_id'].unique()

<StringArray>
['156', '3D1', '152', '002', '148', '149', '075', '0D0', '0D1', '0D2', '070',
 '080', '0D3', '0D4', '140', '141', '144', '150', '371', '372', '576', '280',
 '376', '160', '161', '162', '163', '164', '22A', '6FB', '25C', '360', '361',
 '362', '6FC', '368', '370', '282', '6D0', '325', '6F9', '6CF', '374', '154',
 '375', '660', '378', '6D1', '281', '6FA', '767', '76F', '7C2', '7CA']
Length: 54, dtype: str

In [6]:
import os
import pandas as pd

# Directory path containing CSV files
# /home/hieutt/UniCon/data/can-ml/2017-subaru-forester/merged
input_dir = '../data/can-ml/can-ml/2017-subaru-forester/merged'  # Change the path if needed

# List of CSV files in the directory
files = [f for f in os.listdir(input_dir) if f.endswith('.csv')]

# Function to check each file
def check_data(file_path):
    print(f"\nChecking file: {file_path}")
    
    # Read the file
    df = pd.read_csv(file_path)
    
    # Display the shape of the DataFrame
    print("Shape:", df.shape)
    
    # Display the first 5 rows
    print("Head:")
    display(df.head())
    
    # Check for NaN values
    nan_count = df.isna().sum().sum()
    if nan_count > 0:
        print("Warning: There are NaN values in the data.")
        print(df.isna().sum())
    else:
        print("No NaN values found.")
    
    # Check data types
    print("\nData types:")
    print(df.dtypes)
    
    # Check label distribution if 'label' column exists
    if 'label' in df.columns:
        print("\nLabel distribution:")
        print(df['label'].value_counts())
    
    # Descriptive statistics for numeric columns
    print("\nDescriptive statistics for numeric columns:")
    print(df.describe())
    
# Iterate over each file and check
for file in files:
    check_data(os.path.join(input_dir, file))



Checking file: ../data/can-ml/can-ml/2017-subaru-forester/merged/combined.csv
Shape: (5806825, 4)
Head:


,timestamp,arbitration_id,data_field,attack
0,1.672531e+09,356,1000000000000000000000000000000000000000000000...,0
1,1.672531e+09,881,1100000011111110000111111111111100100000,0
2,1.672531e+09,328,1011000101000101000010000010010000010000000000...,0
3,1.672531e+09,329,1111110000000010011111110011001101010100011000...,0
4,1.672531e+09,117,10000000011000011110,0


No NaN values found.

Data types:
timestamp         float64
arbitration_id      int64
data_field         object
attack              int64
dtype: object

Descriptive statistics for numeric columns:
          timestamp  arbitration_id        attack
count  5.806825e+06    5.806825e+06  5.806825e+06
mean   1.672531e+09    4.111147e+02  1.458026e-02
std    1.636970e+02    3.333655e+02  1.198652e-01
min    1.672531e+09    2.000000e+00  0.000000e+00
25%    1.672531e+09    2.110000e+02  0.000000e+00
50%    1.672531e+09    3.280000e+02  0.000000e+00
75%    1.672532e+09    3.560000e+02  0.000000e+00
max    1.672532e+09    1.994000e+03  1.000000e+00

Checking file: ../data/can-ml/can-ml/2017-subaru-forester/merged/DoS.csv
Shape: (2297897, 4)
Head:


,timestamp,arbitration_id,data_field,attack
0,1.672531e+09,342,1100000000000000000000000000000000000000000000...,0
1,1.672531e+09,320,1001010100000001001110100101101001011111111001...,0
2,1.672531e+09,321,1110100011000010000111001011111010100011001001...,0
3,1.672531e+09,864,1000000000101001000100110000011001100010111000...,0
4,1.672531e+09,865,1110100000000011010100000010000000000000000000...,0


No NaN values found.

Data types:
timestamp         float64
arbitration_id      int64
data_field         object
attack              int64
dtype: object

Descriptive statistics for numeric columns:
          timestamp  arbitration_id        attack
count  2.297897e+06    2.297897e+06  2.297897e+06
mean   1.672532e+09    4.054529e+02  2.254148e-02
std    2.930624e+02    3.377873e+02  1.484364e-01
min    1.672531e+09    0.000000e+00  0.000000e+00
25%    1.672531e+09    2.100000e+02  0.000000e+00
50%    1.672531e+09    3.280000e+02  0.000000e+00
75%    1.672532e+09    3.560000e+02  0.000000e+00
max    1.672532e+09    1.994000e+03  1.000000e+00

Checking file: ../data/can-ml/can-ml/2017-subaru-forester/merged/speed.csv
Shape: (3672497, 4)
Head:


,timestamp,arbitration_id,data_field,attack
0,1.672531e+09,640,1000000011000010000000000000000000000000000000...,0
1,1.672531e+09,117,10000110011011010,0
2,1.672531e+09,208,1111111100000000000000000000000000000000000000...,0
3,1.672531e+09,112,1010000000000000001111111100010010100000000000...,0
4,1.672531e+09,128,1011000000000000000000000000101000101111110000...,0


... (9 outputs left out)

,timestamp,arbitration_id,data_field,attack
0,1.672531e+09,320,1001001000000100000101000001010000010011100100000,0
1,1.672531e+09,321,1110001100101010001111110010101100100100100010...,0
2,1.672531e+09,324,1101110111001110000011000010101000000000110100...,0
3,1.672531e+09,328,1011000100000010000110001000010000010000000001...,0
4,1.672531e+09,329,1011000001100001111110010100110101010000111100...,0


No NaN values found.

Data types:
timestamp         float64
arbitration_id      int64
data_field         object
attack              int64
dtype: object

Descriptive statistics for numeric columns:
          timestamp  arbitration_id        attack
count  3.397184e+06    3.397184e+06  3.397184e+06
mean   1.672531e+09    4.165107e+02  6.579567e-03
std    1.962385e+02    3.403088e+02  8.084725e-02
min    1.672531e+09    0.000000e+00  0.000000e+00
25%    1.672531e+09    2.110000e+02  0.000000e+00
50%    1.672531e+09    3.280000e+02  0.000000e+00
75%    1.672532e+09    3.560000e+02  0.000000e+00
max    1.672532e+09    2.047000e+03  1.000000e+00

Checking file: ../data/can-ml/can-ml/2017-subaru-forester/merged/standstill.csv
Shape: (2361472, 4)
Head:


,timestamp,arbitration_id,data_field,attack
0,1.672531e+09,881,1000000000000000100000000000001101010001111111...,0
1,1.672531e+09,328,1011000100001010100110000110010000010000000001...,0
2,1.672531e+09,329,1011000001100001111110010001010101001000110000...,0
3,1.672531e+09,320,1101000001001101010010000011000000000000110100...,0
4,1.672531e+09,321,1000000100101001101110100010100010101001000001...,0


No NaN values found.

Data types:
timestamp         float64
arbitration_id      int64
data_field         object
attack              int64
dtype: object

Descriptive statistics for numeric columns:
          timestamp  arbitration_id        attack
count  2.361472e+06    2.361472e+06  2.361472e+06
mean   1.672531e+09    4.128220e+02  4.576806e-03
std    1.523671e+02    3.345799e+02  6.749712e-02
min    1.672531e+09    2.000000e+00  0.000000e+00
25%    1.672531e+09    2.110000e+02  0.000000e+00
50%    1.672531e+09    3.280000e+02  0.000000e+00
75%    1.672532e+09    3.560000e+02  0.000000e+00
max    1.672532e+09    1.994000e+03  1.000000e+00
